# Epsilon Fund — Strategy Testing
---

In [2]:
import pandas as pd
import numpy as np
import sys

# ── Set your repo root path ────────────────────────────────────────────────────
ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  
# ──────────────────────────────────────────────────────────────────────────────

sys.path.append(ROOT + r'\infrastructure\data')
sys.path.append(ROOT + r'\infrastructure\backtester')

from binance_client import get_binance_client, get_data, get_multiple_data
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: `365` (1y) · `730` (2y) · `1825` (5y) · `2555` (7y, recommended minimum)

In [3]:
# universe + data (broad candidate history; final top-50 universe chosen inside each fold)

import pandas as pd
import numpy as np

INTERVAL = "1d"
START = "2022-01-01"
MIN_ROWS = 300   # broad prefilter only; stricter history check happens inside each fold

client = get_binance_client()

tickers = client.get_ticker()
tickers_df = pd.DataFrame(tickers)

tickers_df = tickers_df[tickers_df["symbol"].str.endswith("USDT")].copy()

bad_suffix = ("UPUSDT", "DOWNUSDT", "BULLUSDT", "BEARUSDT")
tickers_df = tickers_df[~tickers_df["symbol"].str.endswith(bad_suffix)].copy()

bad_exact = {
    "USDCUSDT", "USDTUSDT", "BUSDUSDT", "FDUSDUSDT",
    "TUSDUSDT", "DAIUSDT", "EURUSDT", "USDPUSDT"
}
candidates = [s for s in tickers_df["symbol"].tolist() if s not in bad_exact]

closes = {}
volumes = []
kept = []

for s in candidates:
    try:
        raw = client.get_historical_klines(s, INTERVAL, START)
    except Exception:
        continue

    if len(raw) < MIN_ROWS:
        continue

    df_s = pd.DataFrame(
        raw,
        columns=[
            "Time", "Open", "High", "Low", "Close", "Volume",
            "Close_time", "Quote_volume", "Trades",
            "Taker_base", "Taker_quote", "Ignore"
        ]
    )

    df_s = df_s[["Time", "Close", "Volume"]].copy()
    df_s["Time"] = pd.to_datetime(df_s["Time"], unit="ms")
    df_s["Close"] = df_s["Close"].astype(float)
    df_s["Volume"] = df_s["Volume"].astype(float)
    df_s = df_s.set_index("Time").sort_index()

    closes[s] = df_s["Close"]
    volumes.append(df_s["Volume"].rename(s))
    kept.append(s)

    print("loaded", s, len(df_s))

close_df_all = pd.concat(closes, axis=1).sort_index()
volume_df_all = pd.concat(volumes, axis=1).sort_index()

print("broad candidate symbols:", len(kept))
print("close_df_all shape:", close_df_all.shape)
print("volume_df_all shape:", volume_df_all.shape)
print(kept[:20])

loaded BTCUSDT 1554
loaded ETHUSDT 1554
loaded BNBUSDT 1554
loaded NEOUSDT 1554
loaded LTCUSDT 1554
loaded QTUMUSDT 1554
loaded ADAUSDT 1554
loaded XRPUSDT 1554
loaded EOSUSDT 1242
loaded IOTAUSDT 1554
loaded XLMUSDT 1554
loaded ONTUSDT 1554
loaded TRXUSDT 1554
loaded ETCUSDT 1554
loaded ICXUSDT 1554
loaded NULSUSDT 1202
loaded VETUSDT 1554
loaded LINKUSDT 1554
loaded WAVESUSDT 899
loaded ONGUSDT 1554
loaded HOTUSDT 1554
loaded ZILUSDT 1554
loaded ZRXUSDT 1554
loaded FETUSDT 1554
loaded BATUSDT 1554
loaded XMRUSDT 781
loaded ZECUSDT 1554
loaded IOSTUSDT 1554
loaded CELRUSDT 1554
loaded DASHUSDT 1554
loaded OMGUSDT 899
loaded THETAUSDT 1554
loaded ENJUSDT 1554
loaded MITHUSDT 356
loaded MATICUSDT 984
loaded ATOMUSDT 1554
loaded TFUELUSDT 1554
loaded ONEUSDT 1554
loaded FTMUSDT 1109
loaded ALGOUSDT 1554
loaded GTOUSDT 332
loaded DOGEUSDT 1554
loaded DUSKUSDT 1554
loaded ANKRUSDT 1554
loaded WINUSDT 1554
loaded COSUSDT 1554
loaded COCOSUSDT 514
loaded MTLUSDT 1554
loaded TOMOUSDT 689
load

---
## Strategy

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> Signals are shifted 1 bar by the engine — no need to shift manually.

In [9]:
import numpy as np
import pandas as pd
from itertools import combinations
from statsmodels.tsa.stattools import adfuller

# =========================================================
# CHECK DATA EXISTS
# =========================================================
print("close_df_all shape:", close_df_all.shape)
print("volume_df_all shape:", volume_df_all.shape)

# =========================================================
# PARAMETERS
# =========================================================
lookback = 126
z_lookback = 60
entry = 2.0
exit = 0.5
stop_z = 4.0
max_holding = 15

TOP_N_COINS = 25

MIN_HISTORY = 350
MIN_TRAIN_OBS = 120
MIN_TRADES = 4

MIN_RET_CORR = 0.2
MAX_VOL = 0.08

MIN_HALF_LIFE = 2
MAX_HALF_LIFE = 40

MAX_BETA_STD = 0.75
MIN_BETA_MED = -2.5
MAX_BETA_MED = 2.5

MAX_SIMPLE_DD = -0.35
MIN_SIMPLE_PF = 1.0
MIN_SIMPLE_RETURN = 0.0

N_SUBPERIODS = 4
MIN_GOOD_SUBPERIODS = 2

# =========================================================
# STEP 1: LIQUID UNIVERSE
# =========================================================
avg_dollar_vol = (close_df_all * volume_df_all).mean(axis=0, skipna=True)
hist_count = close_df_all.notna().sum()
volatility = close_df_all.pct_change().std()

eligible = (
    (hist_count > 800) &
    (volatility < MAX_VOL)
)

symbols = avg_dollar_vol[eligible].sort_values(ascending=False).head(TOP_N_COINS).index.tolist()

print(f"Using {len(symbols)} filtered liquid coins")
print(symbols)

# =========================================================
# HELPERS
# =========================================================
def estimate_half_life(spread: pd.Series) -> float:
    s = spread.dropna()
    if len(s) < 50:
        return np.nan

    lagged = s.shift(1)
    delta = s.diff()

    reg_df = pd.concat([lagged.rename("lagged"), delta.rename("delta")], axis=1).dropna()
    if len(reg_df) < 30:
        return np.nan

    x = reg_df["lagged"].values
    y = reg_df["delta"].values

    try:
        beta = np.polyfit(x, y, 1)[0]
    except Exception:
        return np.nan

    if beta >= 0:
        return np.inf

    hl = -np.log(2) / beta
    return hl


def build_pair(price_df: pd.DataFrame, Y: str, X: str):
    df = price_df[[Y, X]].dropna().copy()
    if len(df) < MIN_HISTORY:
        return None

    # return-correlation filter
    ret_corr = df[Y].pct_change().corr(df[X].pct_change())
    if pd.isna(ret_corr) or ret_corr < MIN_RET_CORR:
        return None

    logY = np.log(df[Y])
    logX = np.log(df[X])

    beta = logY.rolling(lookback).cov(logX) / logX.rolling(lookback).var()
    alpha = logY.rolling(lookback).mean() - beta * logX.rolling(lookback).mean()

    spread = logY - (alpha + beta * logX)
    spread_mean = spread.rolling(z_lookback).mean()
    spread_std = spread.rolling(z_lookback).std()
    z = (spread - spread_mean) / spread_std

    # beta stability filters
    beta_clean = beta.dropna()
    if len(beta_clean) < MIN_TRAIN_OBS:
        return None

    beta_std = beta_clean.std()
    beta_med = beta_clean.median()

    if pd.isna(beta_std) or pd.isna(beta_med):
        return None
    if beta_std > MAX_BETA_STD:
        return None
    if not (MIN_BETA_MED <= beta_med <= MAX_BETA_MED):
        return None

    # half-life filter
    half_life = estimate_half_life(spread)
    if not np.isfinite(half_life) or half_life < MIN_HALF_LIFE or half_life > MAX_HALF_LIFE:
        return None

    # simple strategy logic for screening
    raw_signal = pd.Series(np.nan, index=df.index)
    raw_signal[z > entry] = -1.0
    raw_signal[z < -entry] = 1.0
    raw_signal[z.abs() < exit] = 0.0
    raw_signal[z > stop_z] = 0.0
    raw_signal[z < -stop_z] = 0.0

    pos = pd.Series(0.0, index=df.index)
    current_pos = 0.0
    holding_days = 0

    for i in range(len(df)):
        sig = raw_signal.iloc[i]

        if current_pos == 0.0:
            if pd.notna(sig) and sig != 0.0:
                current_pos = sig
                holding_days = 1
            else:
                current_pos = 0.0
                holding_days = 0
        else:
            exit_now = False

            if pd.notna(sig) and sig == 0.0:
                exit_now = True
            if holding_days >= max_holding:
                exit_now = True

            if exit_now:
                current_pos = 0.0
                holding_days = 0
            else:
                holding_days += 1

        pos.iloc[i] = current_pos

    retY = logY.diff()
    retX = logX.diff()
    pair_log_ret = retY - beta.shift(1) * retX
    pair_arith_ret = np.exp(pair_log_ret) - 1.0

    df_out = pd.DataFrame(index=df.index)
    df_out["spread"] = spread
    df_out["z"] = z
    df_out["beta"] = beta.shift(1)
    df_out["position"] = pos
    df_out["pair_ret"] = pair_arith_ret

    df_out = df_out.dropna(subset=["spread", "z", "beta", "pair_ret"])

    if len(df_out) < MIN_TRAIN_OBS:
        return None

    # count flat -> non-flat transitions
    entry_mask = (
        df_out["position"].fillna(0).ne(0) &
        df_out["position"].shift(1).fillna(0).eq(0)
    )
    num_trades = int(entry_mask.sum())

    if num_trades < MIN_TRADES:
        return None

    # simple screening metrics
    strat_ret = df_out["position"].shift(1).fillna(0) * df_out["pair_ret"]
    equity = (1.0 + strat_ret.fillna(0.0)).cumprod()

    total_return = equity.iloc[-1] - 1.0
    running_max = equity.cummax()
    drawdown = equity / running_max - 1.0
    max_dd = drawdown.min()

    pnl_by_trade = []
    pos_series = df_out["position"].fillna(0)
    eq_series = equity

    in_trade = False
    entry_idx = None

    for i in range(1, len(df_out)):
        prev_pos = pos_series.iloc[i - 1]
        curr_pos = pos_series.iloc[i]

        if (prev_pos == 0) and (curr_pos != 0):
            in_trade = True
            entry_idx = i

        elif in_trade and (prev_pos != 0) and (curr_pos == 0):
            trade_ret = eq_series.iloc[i] / eq_series.iloc[entry_idx - 1] - 1.0 if entry_idx > 0 else eq_series.iloc[i] - 1.0
            pnl_by_trade.append(trade_ret)
            in_trade = False
            entry_idx = None

    wins = [x for x in pnl_by_trade if x > 0]
    losses = [x for x in pnl_by_trade if x < 0]

    gross_profit = sum(wins) if wins else 0.0
    gross_loss = abs(sum(losses)) if losses else 0.0
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf

    # subperiod stability
    good_subperiods = 0
    split_frames = np.array_split(df_out, N_SUBPERIODS)

    for sub_df in split_frames:
        if len(sub_df) < max(MIN_TRAIN_OBS // 2, 40):
            continue

        sub_pos = sub_df["position"].fillna(0)
        sub_ret = sub_pos.shift(1).fillna(0) * sub_df["pair_ret"]
        sub_eq = (1.0 + sub_ret.fillna(0.0)).cumprod()

        sub_total_return = sub_eq.iloc[-1] - 1.0
        sub_dd = (sub_eq / sub_eq.cummax() - 1.0).min()

        if (sub_total_return > -0.05) and (sub_dd > -0.25):
            good_subperiods += 1

    if total_return <= MIN_SIMPLE_RETURN:
        return None
    if max_dd < MAX_SIMPLE_DD:
        return None
    if profit_factor < MIN_SIMPLE_PF:
        return None
    if good_subperiods < MIN_GOOD_SUBPERIODS:
        return None

    return {
        "pair_df": df_out,
        "ret_corr": ret_corr,
        "half_life": half_life,
        "beta_std": beta_std,
        "beta_med": beta_med,
        "num_trades": num_trades,
        "total_return": total_return,
        "max_dd": max_dd,
        "profit_factor": profit_factor,
        "good_subperiods": good_subperiods,
    }

# =========================================================
# STEP 2: RUN SCREEN
# =========================================================
pairs = list(combinations(symbols, 2))
print(f"Total pairs to test: {len(pairs)}")

candidates = []

for i, (Y, X) in enumerate(pairs, 1):
    if i % 50 == 0 or i == len(pairs):
        print(f"Processed {i}/{len(pairs)} pairs")

    result = build_pair(close_df_all, Y, X)
    if result is None:
        continue

    pair_df = result["pair_df"]
    spread = pair_df["spread"].dropna()

    if len(spread) < MIN_TRAIN_OBS:
        continue

    try:
        _, pval, _, _, _, _ = adfuller(spread.values)
    except Exception:
        continue

    if pval >= 0.1:
        continue

    score = (
        2.0 * (-np.log10(max(pval, 1e-12))) +
        1.5 * result["ret_corr"] +
        1.0 * min(result["num_trades"], 20) / 20.0 +
        1.5 * min(result["profit_factor"], 5.0) / 5.0 +
        1.0 * result["good_subperiods"] / N_SUBPERIODS +
        1.0 * max(0.0, 1.0 - abs(result["max_dd"]) / 0.35) +
        1.0 * max(0.0, 1.0 - (result["half_life"] - MIN_HALF_LIFE) / (MAX_HALF_LIFE - MIN_HALF_LIFE))
    )

    candidates.append({
        "pair": f"{Y}__{X}",
        "Y": Y,
        "X": X,
        "adf_p": pval,
        "ret_corr": result["ret_corr"],
        "half_life": result["half_life"],
        "beta_med": result["beta_med"],
        "beta_std": result["beta_std"],
        "num_trades": result["num_trades"],
        "simple_return": result["total_return"],
        "simple_max_dd": result["max_dd"],
        "simple_pf": result["profit_factor"],
        "good_subperiods": result["good_subperiods"],
        "screen_score": score,
    })

# =========================================================
# STEP 3: RANK
# =========================================================
candidates_df = pd.DataFrame(candidates)

if len(candidates_df) == 0:
    print("No candidates passed the filters.")
else:
    candidates_df = candidates_df.sort_values(
        ["screen_score", "adf_p", "num_trades"],
        ascending=[False, True, False]
    ).reset_index(drop=True)

    print("\nTOP 20 PAIRS")
    display(candidates_df.head(20))

    top_pairs = candidates_df.head(20)["pair"].tolist()
    pair_list = [tuple(p.split("__")) for p in top_pairs]

    print("\nTOP 20 CANDIDATE PAIRS")
    for p in pair_list:
        print(p)

close_df_all shape: (1554, 493)
volume_df_all shape: (1554, 493)
Using 25 filtered liquid coins
['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT', 'DOGEUSDT', 'BNBUSDT', 'PEPEUSDT', 'SUIUSDT', 'ADAUSDT', 'SHIBUSDT', 'AVAXUSDT', 'MATICUSDT', 'TRXUSDT', 'FTMUSDT', 'ARBUSDT', 'LINKUSDT', 'WLDUSDT', 'BONKUSDT', 'NEARUSDT', 'LTCUSDT', 'GMTUSDT', 'ORDIUSDT', 'DOTUSDT', 'GALAUSDT', 'APTUSDT']
Total pairs to test: 300


C:\Users\user\AppData\Local\Temp\ipykernel_45920\1876367962.py:50: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  volatility = close_df_all.pct_change().std()
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please 

Processed 50/300 pairs


c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnum

Processed 100/300 pairs


...

c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnum

Processed 300/300 pairs

TOP 20 PAIRS


c:\Users\user\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


,pair,Y,X,adf_p,ret_corr,half_life,beta_med,beta_std,num_trades,simple_return,simple_max_dd,simple_pf,good_subperiods,screen_score
0,TRXUSDT__LINKUSDT,TRXUSDT,LINKUSDT,7.864345e-07,0.325096,9.012456,0.290109,0.311508,33,2.013637,-0.184838,4.514275,4,17.337954
1,TRXUSDT__DOTUSDT,TRXUSDT,DOTUSDT,3.091585e-07,0.359670,10.070034,0.211058,0.310184,35,0.147040,-0.295018,1.269223,2,16.384633
2,AVAXUSDT__NEARUSDT,AVAXUSDT,NEARUSDT,7.099131e-06,0.750946,18.621692,0.887655,0.309060,34,4.887341,-0.242344,4.012630,3,15.247973
3,TRXUSDT__GALAUSDT,TRXUSDT,GALAUSDT,2.105380e-05,0.348867,10.767989,0.110823,0.212241,32,0.658460,-0.288333,1.950243,2,12.907167
4,BTCUSDT__SOLUSDT,BTCUSDT,SOLUSDT,4.987268e-05,0.746371,25.838920,0.462397,0.214340,33,0.170680,-0.337569,1.280566,3,12.266179
5,TRXUSDT__GMTUSDT,TRXUSDT,GMTUSDT,4.247710e-05,0.262029,13.949744,0.086453,0.243426,32,0.779994,-0.310792,2.006033,2,12.036100
6,TRXUSDT__APTUSDT,TRXUSDT,APTUSDT,1.141564e-04,0.270965,13.071127,0.066383,0.271560,32,1.369307,-0.275781,2.913078,3,11.836078
7,BNBUSDT__FTMUSDT,BNBUSDT,FTMUSDT,1.736659e-04,0.607206,16.639478,0.285135,0.169771,24,0.582912,-0.333862,1.955964,3,11.429030
8,ETHUSDT__SOLUSDT,ETHUSDT,SOLUSDT,2.894926e-04,0.744749,28.272271,0.600672,0.398139,31,0.383074,-0.331884,1.495804,2,10.502975
9,TRXUSDT__LTCUSDT,TRXUSDT,LTCUSDT,2.004208e-03,0.340811,10.610096,0.209659,0.423639,28,0.573834,-0.323322,1.983702,3,9.102081



TOP 20 CANDIDATE PAIRS
('TRXUSDT', 'LINKUSDT')
('TRXUSDT', 'DOTUSDT')
('AVAXUSDT', 'NEARUSDT')
('TRXUSDT', 'GALAUSDT')
('BTCUSDT', 'SOLUSDT')
('TRXUSDT', 'GMTUSDT')
('TRXUSDT', 'APTUSDT')
('BNBUSDT', 'FTMUSDT')
('ETHUSDT', 'SOLUSDT')
('TRXUSDT', 'LTCUSDT')
('SHIBUSDT', 'BONKUSDT')
('MATICUSDT', 'APTUSDT')
('SHIBUSDT', 'ORDIUSDT')


---
## Backtest

| Parameter | Options | Default |
|---|---|---|
| `cost` | Cost per trade as decimal — `0.001` = 0.1% | `0.0` |
| `show_plot` | `True` / `False` | `True` |
| `save_html` | Filename string or `None` | `None` |
| `show_trades` | Overlay entry/exit markers on price chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | same asset |